# csp_workflow_mp — visualise benchmark predictions

Diagnostic figures for the raw output written by `scripts/05_run_benchmark.py`:

1. Per-stage success rate for each retrieval strategy (substitution → relaxation → SG match → StructureMatcher match).
2. RMSD distribution on the ordered subset.
3. Space-group match rate stratified by the number of constituent elements.
4. Volume-change distribution before / after MatterSim relaxation.

The notebook reads whatever strategies are present in `results/benchmark_raw.csv`. A default run of `05_run_benchmark.py` produces `unconstrained` + `sg_only`; passing `--k 1 --k 3 --k 10` in separate runs adds the corresponding rows to the same CSV. Set `CSP_RESULTS_DIR` if the raw CSV lives outside `results/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Canonical results directory (respects CSP_RESULTS_DIR)
from csp_workflow_mp._paths import RESULTS_DIR

raw_csv = RESULTS_DIR / 'benchmark_raw.csv'
df = pd.read_csv(raw_csv)
strategies = sorted(df['strategy'].unique())
print(f'{len(df)} rows  |  strategies: {strategies}')
df.head()

## 1. Per-stage success rate

Each pipeline stage narrows the target population. Space-group (SG) match is the paper's primary structural-fidelity metric; StructureMatcher (SM) is shown alongside as a stricter binary check that requires exact per-site species agreement.

In [ ]:
stages = ['sub_success', 'relax_converged', 'sg_match', 'sm_match']
rates  = (df.groupby('strategy')[stages].mean() * 100).round(1)
rates

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
rates.T.plot(kind='bar', ax=ax, width=0.78)
ax.set_ylabel('Success rate (% of all 500 targets)')
ax.set_xlabel('Pipeline stage')
ax.set_title('Cumulative success at each stage')
ax.set_xticklabels(['Substitution', 'Relax', 'SG match', 'SM match'], rotation=0)
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
ax.legend(title='Strategy', frameon=False)
plt.tight_layout()
plt.show()

## 2. RMSD distribution on the ordered subset

RMSD is `pymatgen.analysis.structure_matcher.StructureMatcher.get_rms_dist`, computed only for samples where substitution succeeded, relaxation converged, and the ordered predictions could be aligned against the ordered reference.

In [ ]:
rmsd_df = df.dropna(subset=['rmsd_angstrom'])
rmsd_df.groupby('strategy')['rmsd_angstrom'].describe().round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.linspace(0, max(0.5, rmsd_df.rmsd_angstrom.quantile(0.95)), 30)
for strategy, sub in rmsd_df.groupby('strategy'):
    ax.hist(sub.rmsd_angstrom, bins=bins, alpha=0.45, label=strategy)
ax.set_xlabel('RMSD (Å)')
ax.set_ylabel('Count')
ax.set_title('RMSD vs reference, per strategy (ordered subset)')
ax.legend(title='Strategy', frameon=False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. SG match by number of constituent elements

Higher-arity formulas are harder: the template pool gets sparser and the substitution engine has more role assignments to enumerate. The paper stratifies by `n_elements` using space-group match on the valid subset, which is what we plot here.

In [ ]:
# Valid subset only — the same denominator used by the paper's Table 3.
valid = df[df['sub_success'] & df['relax_converged'] & (df['vol_change'] < 0.15)]
by_n = (valid.groupby(['n_elements', 'strategy'])['sg_match']
             .mean().mul(100).round(1)
             .unstack('strategy'))
by_n

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
by_n.plot(kind='bar', ax=ax, width=0.78)
ax.set_xlabel('Number of elements in target formula')
ax.set_ylabel('SG match on valid subset (%)')
ax.set_title('Space-group match by composition complexity')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
ax.legend(title='Strategy', frameon=False)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Volume change during relaxation

`vol_change = |V_relaxed − V_template| / V_template`. Values above the 15 % threshold are marked as filtered by the benchmark and excluded from the valid subset. A cluster of samples with very large ΔV/V usually indicates a template chemically too far from the target for MatterSim to relax cleanly.

In [ ]:
vol_df = df.dropna(subset=['vol_change'])
fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.linspace(0, vol_df.vol_change.quantile(0.99), 30)
for strategy, sub in vol_df.groupby('strategy'):
    ax.hist(sub.vol_change, bins=bins, alpha=0.45, label=strategy)
ax.axvline(0.15, color='k', ls='--', lw=0.8, label='valid-subset threshold (15%)')
ax.set_xlabel('|V_pred − V_template| / V_template')
ax.set_ylabel('Count')
ax.set_title('Volume change distribution, per strategy')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()